<a href="https://colab.research.google.com/github/jason-snow58/Tuning-the-Stability-of-a-Disulfide-Stabilized-Phage-VLP-by-Interface-Guided-Capsid-Engineering/blob/main/Tuning_the_Stability_of_a_Disulfide_Stabilized_Phage_VLP_by_Interface_Guided_Capsid_Engineering_Figure_1_interface_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Figure 1 — interdimer and intradimer contact classification

Supporting analysis for *Tuning the Stability of a Disulfide-Stabilized Phage VLP by Interface-Guided Capsid Engineering*

---

### What this analysis is, in the paper

Coat-protein dimers are the assembly unit of an ssRNA phage capsid, so a mutation that disturbs an **intradimer** contact risks losing assembly altogether, while one that disturbs only an **interdimer** contact can weaken the lattice and leave the dimer intact. Figure 1 is the search for capsids where those two networks overlap least — the ones with the most room to destabilise safely. AP205 wins, with only three residues shared between the two interfaces.

### What the code does

For every residue of a trimer-of-dimers, it measures the side-chain heavy atoms against every heavy atom of the neighbouring subunits at a 4.0 Å cutoff, works out which subunit pairs are the dimers, and labels each residue interdimer, intradimer, or both.

Two details decide the numbers and are easy to get wrong. Subunits are told apart by **segment identifier**, not chain: in AP205, chains LE, ME and NE all carry chain letter `F` while belonging to three different dimers. And a residue counts as an interface residue only if it contributes a **side chain** — mutation acts on side chains, so a residue meeting an interface with backbone alone is not a target.

### How it is organised

Four phases, one cell each: **decide → render → validate → effect**. The boundary is real rather than decorative — `effect` does not import `render`, and never receives an analysis record.

## Setup

In [ ]:
#@title Setup and input controls { display-mode: "form" }
#@markdown Run this cell first. It detects the environment, installs anything
#@markdown missing, imports everything the notebook needs, and collects the
#@markdown input settings below. Defaults reproduce the published analysis.

# ---- environment -----------------------------------------------------------
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import glob
import os
import subprocess
import sys

def ensure(package, module=None):
    """Import a package, installing it first if this is Colab."""
    name = module or package
    try:
        __import__(name)
        return True
    except ImportError:
        if IN_COLAB:
            print(f"installing {package} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package],
                           check=True)
            __import__(name)
            return True
        print(f"MISSING: {package}")
        print(f"  conda install -c conda-forge {package}")
        return False


# ---- dependencies ----------------------------------------------------------
READY = all([ensure("numpy"), ensure("MDAnalysis")])
import numpy
import MDAnalysis
print("numpy", numpy.__version__, "| MDAnalysis", MDAnalysis.__version__)

# ---- input controls --------------------------------------------------------
#@markdown **Input.** Leave blank to search `data/models/` and `data/`, or in
#@markdown Colab to be offered an upload box.
PDB_FILE = "" #@param {type:"string"}
#@markdown **Contact cutoff (Å).** 4.0 is the published value.
CUTOFF = 4.0 #@param {type:"number"}
#@markdown **Minimum residue contacts for a subunit pair to count as a dimer.**
#@markdown The published value is 10; results are unchanged from 0 to 120.
MIN_DIMER_CONTACTS = 10 #@param {type:"integer"}
#@markdown **Exclude hydrogens.** The published analysis used heavy atoms only.
EXCLUDE_HYDROGENS = True #@param {type:"boolean"}
#@markdown **Also write the PyMOL script** that builds the interface session.
WRITE_PYMOL_SCRIPT = True #@param {type:"boolean"}
OUTPUT_DIR = "output" #@param {type:"string"}

# ---- helpers ---------------------------------------------------------------
def find_input(description, patterns, search_dirs, allow_multiple=False):
    """Locate an input file, or offer an upload box in Colab.

    Patterns are tried in order, so a specific name wins over a general glob.
    Returns one path unless allow_multiple=True.
    """
    for pattern in patterns:
        hits, seen = [], set()
        for directory in search_dirs:
            # '**' searches subdirectories, which matters because each analysis
            # is committed into its own directory named after the structure.
            found = glob.glob(os.path.join(directory, pattern), recursive=True)
            for path in sorted(found):
                real = os.path.realpath(path)
                if os.path.isfile(path) and real not in seen:
                    seen.add(real)
                    hits.append(path)
        if hits:
            print(f"input: matched {pattern!r}")
            for h in hits:
                print("   ", h)
            if len(hits) > 1 and not allow_multiple:
                print("   using the first; set the variable directly to choose another")
                return hits[0]
            return hits if allow_multiple else hits[0]

    if IN_COLAB:
        from google.colab import files
        print(f"Upload {description}:")
        uploaded = files.upload()
        if not uploaded:
            return None
        names = list(uploaded)
        return names if allow_multiple else names[0]

    print(f"Nothing found for {description}.")
    print("Searched:", ", ".join(search_dirs))
    print("Patterns:", ", ".join(repr(p) for p in patterns))
    return None


def require(value, what):
    if value is None:
        raise SystemExit(f"No input for {what}. See the message above.")
    return value


def deliver(paths):
    """Report finished files, and download them in Colab."""
    paths = [paths] if isinstance(paths, str) else list(paths)
    for p in paths:
        size = os.path.getsize(p) if os.path.exists(p) else 0
        print(f"   {p}  ({size:,} bytes)")
    if IN_COLAB:
        from google.colab import files
        for p in paths:
            files.download(p)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print()
print("Colab" if IN_COLAB else "local Jupyter", "| python", sys.version.split()[0])
print("output directory:", os.path.abspath(OUTPUT_DIR))

## The analysis code

These cells write the analysis package into the working directory, so the notebook is self-contained and nothing has to be fetched. This is the same source that accompanies the manuscript — read it if you want to check the calculation, or run straight past it.

In [ ]:
os.makedirs("capsid", exist_ok=True)
print("package directory ready")

In [ ]:
%%writefile capsid/__init__.py
"""Interface and contact analysis for ssRNA phage capsids.

WHICH PIPELINES ARE SEPARATED, AND WHICH ARE NOT

Only the contact-classification path implements the phase separation in its
call graph:

    decide.analyse(pdb)                  reads the structure; classifies; once
    render.render_analysis(decision)     pure; produces every finished file
    validate.validate_rendering(...)     three derivations compared row by row
    effect.commit_artifacts(...)         transport only; cannot render

``effect`` does not import ``render`` and never receives an analysis record, so
no output path can decide or render once writing has begun. Files are staged
and committed as a set, so a failed run leaves the previous output untouched
rather than a directory holding half of one run and half of another.

The three PyMOL and figure modules -- ``pymol_contacts``, ``calpha_svg`` and
``network_map`` -- are a different shape. Each interleaves reading, deciding,
drawing and writing in a single operation, and each writes files or changes
PyMOL state directly. They are kept that way deliberately, because they
reproduce the retained outputs and restructuring their internals would put that
at risk. The guarantees above describe the classification path only.
"""

import importlib

__all__ = ['artifacts', 'calpha_svg', 'decide', 'effect', 'manifest',
           'network_map', 'palette', 'pymol_contacts', 'records', 'render',
           'validate']


def __getattr__(name):
    """Import submodules on first use.

    Importing the package must not require every optional dependency. A
    notebook that only classifies contacts needs neither matplotlib nor PyMOL,
    and should not be made to install them to say ``import capsid``.
    """
    if name in __all__:
        module = importlib.import_module(f'.{name}', __name__)
        globals()[name] = module
        return module
    raise AttributeError(f'module {__name__!r} has no attribute {name!r}')


def __dir__():
    return sorted(__all__)

In [ ]:
%%writefile capsid/records.py
"""Record types produced by DECIDE and consumed by RENDER and EFFECT.

Nothing here reads a structure and nothing here classifies. These are the
containers that carry the evidence -- the exact atoms and distances behind every
classification -- so that a reader of any output can see why a residue was
assigned as it was.
"""

from dataclasses import dataclass, field
from types import MappingProxyType
from typing import Dict, FrozenSet, List, Mapping, Optional, Set, Tuple

# (chain/segment id, residue number, residue name)
ResidueKey = Tuple[str, int, str]
# (chain, chain) always sorted, so a pair has exactly one spelling
ChainPair = Tuple[str, str]

BACKBONE_ATOMS = frozenset({'N', 'CA', 'C', 'O', 'OXT'})

# --- closed vocabularies -----------------------------------------------------
# Declared as lists, not implied by the order of if/elif branches, so that
# changing precedence is a visible one-line edit.

INTERFACE_TYPES = ('intra-dimer', 'inter-dimer')

RESIDUE_CLASSES = ('both', 'inter-dimer', 'intra-dimer', 'none')

MUTATION_POTENTIALS = ('high', 'moderate', 'none')


def is_backbone(atom_name: str) -> bool:
    return atom_name in BACKBONE_ATOMS


class _ContactViews:
    """Derived views shared by the build-time and frozen contact types.

    Held in a mixin so the two types cannot drift apart: there is one
    definition of what 'this side chain reached' means, not two.
    """

    @property
    def chain_pair(self) -> ChainPair:
        a, b = sorted((self.res1[0], self.res2[0]))
        return (a, b)

    @property
    def res1_sc_found_target(self) -> bool:
        return self.dist_res1_sc_to_res2 != float('inf')

    @property
    def res2_sc_found_target(self) -> bool:
        return self.dist_res2_sc_to_res1 != float('inf')

    @property
    def contact_type(self) -> str:
        """'sc-sc' when a side-chain/side-chain pair exists, else 'sc-bb'."""
        return 'sc-sc' if self.has_sc_sc_contact else 'sc-bb'


@dataclass
class _ContactBuilder(_ContactViews):
    """Mutable during DECIDE only. Frozen into a Contact before RENDER sees it."""
    res1: ResidueKey
    res2: ResidueKey
    min_distance: float = float('inf')
    interface_type: str = ''
    dist_res1_sc_to_res2: float = float('inf')
    atom_res1_sc_name: str = ''
    atom_res2_target_name: str = ''
    dist_res2_sc_to_res1: float = float('inf')
    atom_res2_sc_name: str = ''
    atom_res1_target_name: str = ''
    has_sc_sc_contact: bool = False
    res1_sc_hit_res2_bb: bool = False
    res2_sc_hit_res1_bb: bool = False

    def freeze(self) -> 'Contact':
        return Contact(**vars(self))


def _check_interface_type(value: str) -> None:
    if value not in INTERFACE_TYPES:
        raise ValueError(f'interface_type outside the closed vocabulary: '
                         f'{value!r}')


@dataclass(frozen=True)
class Contact(_ContactViews):
    """One residue-residue contact, with the atoms that produced it.

    Detection is directional: each residue is probed *by its side chain* against
    every heavy atom of the partner. A pair can therefore be seen twice, once
    from each side, and the two views can involve different atoms and different
    distances. Both are kept -- collapsing them to a single minimum is what
    hides a side-chain/side-chain contact behind a closer backbone one.

    ``res1``/``res2`` are stored in sorted order so the pair has one identity.
    """
    res1: ResidueKey
    res2: ResidueKey
    min_distance: float = float('inf')
    interface_type: str = ''            # set by DECIDE, one of INTERFACE_TYPES

    # --- evidence: res1's side chain probing res2 ---
    dist_res1_sc_to_res2: float = float('inf')
    atom_res1_sc_name: str = ''
    atom_res2_target_name: str = ''

    # --- evidence: res2's side chain probing res1 ---
    dist_res2_sc_to_res1: float = float('inf')
    atom_res2_sc_name: str = ''
    atom_res1_target_name: str = ''

    # --- topology flags, set if ANY qualifying atom pair is within the cutoff.
    # Independent of which pair happened to be closest, which is the point.
    has_sc_sc_contact: bool = False
    res1_sc_hit_res2_bb: bool = False
    res2_sc_hit_res1_bb: bool = False

    def __post_init__(self):
        _check_interface_type(self.interface_type)


def classify(intra_partners, inter_partners) -> str:
    """The single definition of the residue classification.

    Called once per residue, at the DECIDE boundary. The result is stored in
    the frozen record; nothing downstream recomputes it.
    """
    has_intra, has_inter = bool(intra_partners), bool(inter_partners)
    if has_intra and has_inter:
        return 'both'
    if has_inter:
        return 'inter-dimer'
    if has_intra:
        return 'intra-dimer'
    return 'none'


def rank_mutation_potential(inter_partners, inter_sc_to_sc) -> str:
    """The single definition of the mutation-potential ranking."""
    if not inter_partners:
        return 'none'
    return 'high' if inter_sc_to_sc > 0 else 'moderate'


class _ResidueViews:
    """Convenience aggregates. These are not decisions and are not reported
    as such; the decided labels are stored fields on the frozen record."""

    @property
    def inter_sc_contacts(self) -> int:
        return self.inter_sc_to_sc + self.inter_sc_to_bb

    @property
    def total_partners(self) -> int:
        return len(self.intra_dimer_partners) + len(self.inter_dimer_partners)

    @property
    def partner_chains(self) -> Set[str]:
        return {p[0] for p in
                set(self.intra_dimer_partners) | set(self.inter_dimer_partners)}


@dataclass
class _ResidueBuilder(_ResidueViews):
    """Mutable during DECIDE only."""
    residue_key: ResidueKey
    intra_dimer_partners: Set[ResidueKey] = field(default_factory=set)
    inter_dimer_partners: Set[ResidueKey] = field(default_factory=set)
    inter_sc_to_sc: int = 0
    inter_sc_to_bb: int = 0
    intra_sc_contacts: int = 0

    def freeze(self) -> 'ResidueSummary':
        """Decide the labels once, here, and hand over a closed record."""
        return ResidueSummary(
            residue_key=self.residue_key,
            intra_dimer_partners=frozenset(self.intra_dimer_partners),
            inter_dimer_partners=frozenset(self.inter_dimer_partners),
            inter_sc_to_sc=self.inter_sc_to_sc,
            inter_sc_to_bb=self.inter_sc_to_bb,
            intra_sc_contacts=self.intra_sc_contacts,
            classification=classify(self.intra_dimer_partners,
                                    self.inter_dimer_partners),
            mutation_potential=rank_mutation_potential(
                self.inter_dimer_partners, self.inter_sc_to_sc))


@dataclass(frozen=True)
class ResidueSummary(_ResidueViews):
    """Per-residue roll-up. Only residues that contribute a SIDE CHAIN appear.

    A residue that touches an interface with backbone only is not a mutation
    target, so it is deliberately not summarised.

    NOTE ON THE COUNTER FIELDS -- read before using them.
    ``inter_sc_to_sc`` and ``inter_sc_to_bb`` count side-chain-mediated contacts
    of either interface type, not interdimer contacts alone, and
    ``intra_sc_contacts`` is always 0. These fields are informational only.
    The residue classification, and every count reported in the manuscript, is
    derived from the partner sets below and is unaffected by them.
    """
    residue_key: ResidueKey
    intra_dimer_partners: FrozenSet[ResidueKey] = frozenset()
    inter_dimer_partners: FrozenSet[ResidueKey] = frozenset()
    inter_sc_to_sc: int = 0
    inter_sc_to_bb: int = 0
    intra_sc_contacts: int = 0          # always 0; see the note above

    # Decided at the DECIDE boundary and stored, not recomputed on each read.
    classification: str = 'none'
    mutation_potential: str = 'none'

    def __post_init__(self):
        if self.classification not in RESIDUE_CLASSES:
            raise ValueError(f'classification outside the closed vocabulary: '
                             f'{self.classification!r}')
        if self.mutation_potential not in MUTATION_POTENTIALS:
            raise ValueError(f'mutation_potential outside the closed '
                             f'vocabulary: {self.mutation_potential!r}')
        # The stored label must agree with the evidence it was decided from.
        expected = classify(self.intra_dimer_partners, self.inter_dimer_partners)
        if self.classification != expected:
            raise ValueError(
                f'{self.residue_key}: stored classification '
                f'{self.classification!r} contradicts its partner sets '
                f'({expected!r})')


@dataclass(frozen=True)
class InterfaceAnalysis:
    """Everything one run decided. RENDER and EFFECT read only this.

    Built once, in decide.analyse(). After it is returned, no output path may
    reopen the structure or recompute a classification -- and now none can
    quietly edit the answer either: the record is frozen, its contacts and
    residues are frozen, and its mappings are read-only views. The phase
    boundary is enforced by the type rather than asserted in a comment.
    """
    pdb_path: str
    pdb_sha256: str          # identity of the exact coordinates decided on
    cutoff: float
    exclude_hydrogens: bool
    min_contacts_threshold: int
    chain_ids: Tuple[str, ...]

    # insertion-ordered; the CSV row order depends on it
    contacts: Mapping[Tuple[ResidueKey, ResidueKey], Contact]
    residues: Mapping[ResidueKey, ResidueSummary]

    # evidence for the interface call
    atom_contact_counts: Mapping[Tuple[str, str], int]
    chain_pair_counts: Mapping[ChainPair, int]
    best_partner: Mapping[str, Tuple[str, int]]
    intra_dimer_pairs: FrozenSet[ChainPair]

    @staticmethod
    def freeze_mapping(mapping) -> Mapping:
        """A read-only view that preserves insertion order."""
        return MappingProxyType(dict(mapping))

    # ---- queries: pure lookups over the records above, never re-derivation ----

    def dimer_partner(self, chain: str) -> Optional[str]:
        hit = self.best_partner.get(chain)
        return hit[0] if hit else None

    def by_class(self, classification: str) -> List[ResidueSummary]:
        return [s for s in self.residues.values()
                if s.classification == classification]

    def by_potential(self, potential: str) -> List[ResidueSummary]:
        return [s for s in self.residues.values()
                if s.mutation_potential == potential]

    def resids_by_class(self, classification: str) -> Dict[str, Set[int]]:
        out: Dict[str, Set[int]] = {}
        for s in self.residues.values():
            if s.classification == classification:
                out.setdefault(s.residue_key[0], set()).add(s.residue_key[1])
        return out

    def dimer_partner_resids(self) -> Dict[str, Set[int]]:
        """Residues in a dimer partner chain that a 'both' residue contacts.

        These are drawn green: they are the intra-dimer half of the coupling
        that makes a 'both' residue risky to mutate.
        """
        out: Dict[str, Set[int]] = {}
        for s in self.by_class('both'):
            partner_chain = self.dimer_partner(s.residue_key[0])
            if not partner_chain:
                continue
            for pk in s.intra_dimer_partners:
                if pk[0] == partner_chain:
                    out.setdefault(pk[0], set()).add(pk[1])
        return out

    def class_counts(self) -> Dict[str, int]:
        counts = {c: 0 for c in RESIDUE_CLASSES}
        for s in self.residues.values():
            counts[s.classification] += 1
        return counts

    def contacts_of_type(self, interface_type: str) -> List[Contact]:
        return [c for c in self.contacts.values()
                if c.interface_type == interface_type]

In [ ]:
%%writefile capsid/artifacts.py
"""The objects that cross the RENDER -> EFFECT boundary.

An Artifact is a finished file: a relative name and the exact bytes. Nothing
downstream may reformat it, rename it, or decide whether it belongs in the set.

The point of these types is that EFFECT can be handed a closed tuple and has
nothing left to decide. If a phase can still choose a filename or re-run a
renderer, the boundary is a claim rather than a property of the call graph.
"""

import hashlib
import posixpath
from dataclasses import dataclass
from typing import Mapping, Tuple


@dataclass(frozen=True)
class Artifact:
    """One finished output file."""

    relative_name: str
    media_type: str
    payload: bytes

    def __post_init__(self):
        name = self.relative_name
        if not name or name != posixpath.normpath(name):
            raise ValueError(f'artifact name must be normalised: {name!r}')
        if posixpath.isabs(name) or name.startswith('..') or '\\' in name:
            raise ValueError(f'artifact name must be a safe relative path: {name!r}')

    @property
    def sha256(self) -> str:
        return hashlib.sha256(self.payload).hexdigest()

    @property
    def size(self) -> int:
        return len(self.payload)


@dataclass(frozen=True)
class RenderedAnalysis:
    """Everything RENDER produced for one decision record.

    ``rows`` is the structured view; ``artifacts`` are the finished files;
    ``report`` is the human text. All three come from the same records, and the
    artifact set is closed before any of it reaches EFFECT.
    """

    source: str
    rows: Mapping[str, Tuple[tuple, ...]]
    artifacts: Tuple[Artifact, ...]
    report: str

    def __post_init__(self):
        names = [a.relative_name for a in self.artifacts]
        if len(names) != len(set(names)):
            duplicates = sorted({n for n in names if names.count(n) > 1})
            raise ValueError(f'duplicate artifact names: {duplicates}')

    @property
    def names(self) -> Tuple[str, ...]:
        return tuple(a.relative_name for a in self.artifacts)

    def artifact(self, relative_name: str) -> Artifact:
        for a in self.artifacts:
            if a.relative_name == relative_name:
                return a
        raise KeyError(relative_name)


@dataclass(frozen=True)
class EffectReceipt:
    """What EFFECT was asked to do, what it staged, and what it committed.

    Three separately recorded name/hash sets. They are compared to each other,
    so a commit that silently dropped, added or altered a file cannot report
    success.
    """

    destination: str
    planned: Mapping[str, str]
    staged: Mapping[str, str]
    committed: Mapping[str, str]

    @property
    def consistent(self) -> bool:
        return dict(self.planned) == dict(self.staged) == dict(self.committed)

    def describe(self) -> str:
        lines = [f'destination : {self.destination}',
                 f'files       : {len(self.committed)}',
                 f'consistent  : {self.consistent}']
        for name in sorted(self.committed):
            lines.append(f'  {name}  {self.committed[name][:16]}')
        return '\n'.join(lines)

In [ ]:
%%writefile capsid/manifest.py
"""The closed set of structures the paper reports, declared explicitly.

Enumerating a directory is not an acceptance path. A scratch file dropped into
the models folder would silently widen the analysed set, and a missing paper
structure would silently narrow it -- and a run that compared five of six
structures would still report that everything it checked passed.

So the reproduction commands work from these lists. Anything expected and
absent is an error; anything present and unexpected is an error.
"""

from dataclasses import dataclass
from typing import Tuple


@dataclass(frozen=True)
class Structure:
    stem: str
    phage: str
    pdb_id: str


# The six capsids compared in the manuscript.
PAPER_STRUCTURES: Tuple[Structure, ...] = (
    Structure('5LQP_AP205_3_Fold', 'AP205', '5LQP'),
    Structure('1DWN_PP7_3_Fold', 'PP7', '1DWN'),
    Structure('6YFS_PQ-465_3_fold', 'PQ-465', '6YFS'),
    Structure('7LHD_Qbeta_3_Fold', 'Qbeta', '7LHD'),
    Structure('6YFJ_ESE001_3_Fold', 'ESE001', '6YFJ'),
    Structure('6YFQ_NT-214_3_Fold', 'NT-214', '6YFQ'),
)

# Analysed at the time, retained, and NOT part of the manuscript comparison.
# Kept in a separate list so it can never be counted as a seventh paper capsid.
EXPLORATORY_STRUCTURES: Tuple[Structure, ...] = (
    Structure('6YFR_NT-391_3_Fold', 'NT-391', '6YFR'),
)


def artifact_names(stem: str, include_pml: bool = True,
                   mode=None) -> Tuple[str, ...]:
    """The complete expected artifact set for one structure and contract.

    The tables are identical under both contracts -- the repairs do not touch
    the classification -- so only the PyMOL script carries a mode suffix.
    """
    from .modes import HISTORICAL
    mode = mode or HISTORICAL
    names = [f'{stem}_interface_contacts.csv', f'{stem}_interface_residues.csv']
    if include_pml:
        names.append(mode.named(f'{stem}_interface', 'pml'))
    return tuple(names)


def reference_names(stem: str) -> Tuple[str, ...]:
    """The retained files a reproduction of this structure is compared against."""
    return (f'{stem}_interface_contacts.csv', f'{stem}_interface_residues.csv')


def describe(structures) -> str:
    return ', '.join(f'{s.phage} ({s.pdb_id})' for s in structures)

In [ ]:
%%writefile capsid/modes.py
"""The historical and clean output contracts.

Some of what this code does differs from what the original code did. Each
difference is a repair, and each repair makes the output stop matching the
artifact it is meant to reproduce. Both behaviours are worth having; the
mistake would be to let one quietly stand in for the other.

    HISTORICAL   reproduces the retained artifacts, defects included
    CLEAN        the repaired behaviour, under its own names

A clean artifact never carries a historical filename. A reader who finds
``..._interface.pml`` has the file the paper was built from; one who finds
``..._interface_clean.pml`` has the repaired one, and the name said so before
they opened it.
"""

from dataclasses import dataclass


@dataclass(frozen=True)
class Mode:
    """One output contract."""

    name: str
    summary: str

    # PyMOL scripts: the retained analyzer wrote
    #     hide labels # Hide labels initially to prevent visual clutter
    # PyMOL parses the '#' as a representation name, errors with
    # "unknown representation: '#'", and the labels are never hidden. The
    # historical script keeps the defect because that is what was run.
    pml_inline_comment: bool

    # Appended to artifact names that differ between the two contracts.
    suffix: str

    # Contact detection: 'chain' is the retained invocation. In these models
    # chains LE, ME and NE share PDB chain letter F while belonging to three
    # different dimers, so contacts between them satisfy both the inter and the
    # intra filter and are written twice with contradictory labels. 'segi'
    # keeps the two searches disjoint.
    intra_mode: str

    def named(self, stem: str, extension: str) -> str:
        """Artifact name for this contract, so the two can never collide."""
        return f'{stem}{self.suffix}.{extension}'


HISTORICAL = Mode(
    name='historical',
    summary='reproduces the retained artifacts, defects included',
    pml_inline_comment=True,
    suffix='',
    intra_mode='chain',
)

CLEAN = Mode(
    name='clean',
    summary='repaired behaviour, written under its own names',
    pml_inline_comment=False,
    suffix='_clean',
    intra_mode='segi',
)

MODES = {m.name: m for m in (HISTORICAL, CLEAN)}


def by_name(name: str) -> Mode:
    try:
        return MODES[name]
    except KeyError:
        raise ValueError(f'unknown mode {name!r}; choose from '
                         f'{", ".join(sorted(MODES))}') from None

In [ ]:
%%writefile capsid/palette.py
"""Colour tables shared by every renderer.

These are data, not logic: they are defined once, here, and imported wherever a
colour is needed, so that a PyMOL session and an interactive view cannot drift
apart.
"""

# PyMOL internal colour id -> hex, for py3Dmol / SVG output.
PYMOL_COLOR_MAP = {
    26: '#909090', 5: '#00FFFF', 154: '#FF00FF', 6: '#FFFF00', 9: '#FA8072',
    29: '#FFFFFF', 11: '#708090', 13: '#FF8C00', 10: '#00FF00', 5262: '#008B8B',
    12: '#FF69B4', 36: '#FFD700', 5271: '#8B00FF', 124: '#B3B3B3', 17: '#2E8B57',
    18: '#808000', 5270: '#A0522D', 20: '#008080', 5272: '#9400D3', 52: '#F5DEB3',
    5258: '#E9967A', 5274: '#FFB6C1', 5257: '#7FFFD4', 5256: '#FFFFE0',
    15: '#32CD32', 5277: '#87CEEB', 5279: '#FFC0CB', 5276: '#ADFF2F',
    53: '#EE82EE', 5278: '#F0F8FF', 5275: '#00FA9A', 5269: '#F4A460',
    22: '#228B22', 5266: '#AFEEEE', 5280: '#CD5C5C', 5267: '#9ACD32',
    5268: '#DC143C', 104: '#808080', 23: '#00008B', 51: '#8B4513',
}

# Same ids -> the names PyMOL itself understands, for .pml output.
PYMOL_COLOR_NAMES = {
    26: 'carbon', 5: 'cyan', 154: 'lightmagenta', 6: 'yellow', 9: 'salmon',
    29: 'hydrogen', 11: 'slate', 13: 'orange', 10: 'lime', 5262: 'deepteal',
    12: 'hotpink', 36: 'yelloworange', 5271: 'violetpurple', 124: 'grey70',
    17: 'marine', 18: 'olive', 5270: 'smudge', 20: 'teal', 5272: 'dirtyviolet',
    52: 'wheat', 5258: 'deepsalmon', 5274: 'lightpink', 5257: 'aquamarine',
    5256: 'paleyellow', 15: 'limegreen', 5277: 'skyblue', 5279: 'warmpink',
    5276: 'limon', 53: 'violet', 5278: 'bluewhite', 5275: 'greencyan',
    5269: 'sand', 22: 'forest', 5266: 'lightteal', 5280: 'darksalmon',
    5267: 'splitpea', 5268: 'raspberry', 104: 'grey50', 23: 'deepblue',
    51: 'brown',
}

# Order of PyMOL's ``util.cbc`` chain-colour cycle.
DEFAULT_COLOR_CYCLE = [
    26, 5, 154, 6, 9, 29, 11, 13, 10, 5262, 12, 36, 5271, 124, 17, 18,
    5270, 20, 5272, 52, 5258, 5274, 5257, 5256, 15, 5277, 5279, 5276,
    53, 5278, 5275, 5269, 22, 5266, 5280, 5267, 5268, 104, 23, 51,
]

ALTERNATE_COLOR_CYCLES = {
    'Default (PyMOL CBC)': None,
    'Warm Colors':   [9, 13, 12, 6, 36, 5258, 5279, 5280, 51, 52],
    'Cool Colors':   [5, 17, 20, 23, 5262, 5266, 5277, 5278, 11, 5275],
    'Pastel':        [5256, 5257, 5274, 5277, 5278, 5279, 52, 124, 5266, 5276],
    'High Contrast': [5, 6, 13, 10, 154, 23, 9, 22, 12, 51],
}

# How each residue CLASS is drawn, in both output languages at once, so a
# py3Dmol view and a .pml script can never drift apart.
CLASS_STYLE = {
    'inter-dimer':   {'hex': '#FFA500', 'pymol': 'orange'},
    'both':          {'hex': '#7080B0', 'pymol': 'slate'},
    'dimer_partner': {'hex': '#90B050', 'pymol': 'splitpea'},
}


def chain_colors(chain_ids, color_cycle=None):
    """Map chain id -> {'hex': ..., 'pymol': ...}, cycling the palette."""
    cycle = DEFAULT_COLOR_CYCLE if color_cycle is None else color_cycle
    out = {}
    for i, cid in enumerate(chain_ids):
        key = cycle[i % len(cycle)]
        out[cid] = {'hex': PYMOL_COLOR_MAP.get(key, '#CCCCCC'),
                    'pymol': PYMOL_COLOR_NAMES.get(key, 'grey')}
    return out

In [ ]:
%%writefile capsid/decide.py
"""DECIDE -- the only phase that reads a structure and the only phase that
classifies.

One pass over one structure produces one InterfaceAnalysis record set. Every
output the tool offers is a function of that record set, so the tables, the
PyMOL session and the printed summary cannot disagree with one another or with
the files that get written.

Parameters used for the published analysis: 4.0 A distance cutoff, hydrogens
excluded, and a minimum of 10 residue contacts for a subunit pair to be treated
as a dimer. Running with these settings reproduces the contact and residue
tables reported in the manuscript.
"""

import hashlib
import itertools

import MDAnalysis as mda
from MDAnalysis.analysis.distances import capped_distance

from .records import (InterfaceAnalysis, ResidueKey, _ContactBuilder,
                      _ResidueBuilder, is_backbone)


def _selections(exclude_hydrogens: bool):
    """The two MDAnalysis selections: the probe side and the target side."""
    sidechain = 'protein and not (name N CA C O OXT)'
    whole = 'protein'
    if exclude_hydrogens:
        no_h = ' and not (type H or name H*)'
        sidechain += no_h
        whole += no_h
    return sidechain, whole


def _residue_lookup(atomgroup):
    """Per-atom -> ResidueKey, resolved once for the whole selection."""
    keys = {}
    for res in atomgroup.residues:
        keys[res.resindex] = (res.segment.segid, res.resid, res.resname)
    return [keys[i] for i in atomgroup.resindices]


def analyse(pdb_path: str, cutoff: float = 4.0, exclude_hydrogens: bool = True,
            min_contacts_threshold: int = 10) -> InterfaceAnalysis:
    """Read one structure, decide everything, return the record set.

    After this returns, the structure is not consulted again.
    """
    with open(pdb_path, 'rb') as fh:
        pdb_sha256 = hashlib.sha256(fh.read()).hexdigest()
    universe = mda.Universe(pdb_path)
    chain_ids = tuple(seg.segid for seg in universe.segments)

    # Build in mutable structures, then freeze once. Nothing downstream of this
    # function can edit what was decided.
    state = _State(cutoff=cutoff, exclude_hydrogens=exclude_hydrogens,
                   min_contacts_threshold=min_contacts_threshold)
    _detect_contacts(universe, state)
    _classify_interfaces(state)
    _summarise_residues(state)

    freeze = InterfaceAnalysis.freeze_mapping
    return InterfaceAnalysis(
        pdb_path=pdb_path,
        pdb_sha256=pdb_sha256,
        cutoff=cutoff,
        exclude_hydrogens=exclude_hydrogens,
        min_contacts_threshold=min_contacts_threshold,
        chain_ids=chain_ids,
        contacts=freeze({k: c.freeze() for k, c in state.contacts.items()}),
        residues=freeze({k: r.freeze() for k, r in state.residues.items()}),
        atom_contact_counts=freeze(state.atom_contact_counts),
        chain_pair_counts=freeze(state.chain_pair_counts),
        best_partner=freeze(state.best_partner),
        intra_dimer_pairs=frozenset(state.intra_dimer_pairs),
    )


class _State:
    """Mutable working set, private to DECIDE and discarded when it returns."""

    def __init__(self, cutoff, exclude_hydrogens, min_contacts_threshold):
        self.cutoff = cutoff
        self.exclude_hydrogens = exclude_hydrogens
        self.min_contacts_threshold = min_contacts_threshold
        self.contacts = {}
        self.residues = {}
        self.atom_contact_counts = {}
        self.chain_pair_counts = {}
        self.best_partner = {}
        self.intra_dimer_pairs = set()


# --------------------------------------------------------------------------
# step 1: every side-chain-mediated contact between two different chains
# --------------------------------------------------------------------------

def _detect_contacts(universe, analysis: '_State') -> None:
    """Probe each chain's side chains against every heavy atom of every other.

    Ordered pairs, not unordered: A's side chain reaching B's backbone and B's
    side chain reaching A's backbone are different facts about the same pair,
    and the CSV emits a row for each. Backbone-to-backbone contacts are never
    detected, by construction.
    """
    sidechain_sel, whole_sel = _selections(analysis.exclude_hydrogens)
    contacts = analysis.contacts

    for seg1, seg2 in itertools.permutations(universe.segments, 2):
        probes = seg1.atoms.select_atoms(sidechain_sel)
        targets = seg2.atoms.select_atoms(whole_sel)
        if len(probes) == 0 or len(targets) == 0:
            continue

        pairs, dists = capped_distance(
            probes.positions, targets.positions,
            max_cutoff=analysis.cutoff, return_distances=True)
        if len(pairs) == 0:
            continue

        analysis.atom_contact_counts[(seg1.segid, seg2.segid)] = len(pairs)

        probe_names = list(probes.names)
        target_names = list(targets.names)
        probe_res = _residue_lookup(probes)
        target_res = _residue_lookup(targets)

        for (i, j), dist in zip(pairs, dists):
            r1: ResidueKey = probe_res[i]
            r2: ResidueKey = target_res[j]

            key = (r1, r2) if r1 <= r2 else (r2, r1)
            contact = contacts.get(key)
            if contact is None:
                contact = _ContactBuilder(res1=key[0], res2=key[1])
                contacts[key] = contact

            if dist < contact.min_distance:
                contact.min_distance = dist

            # Topology: does a side-chain/side-chain pair exist at all, and
            # which side reached the other's backbone. Recorded whether or not
            # this pair happens to be the closest one.
            if is_backbone(target_names[j]):
                if contact.res1 == r1:
                    contact.res1_sc_hit_res2_bb = True
                else:
                    contact.res2_sc_hit_res1_bb = True
            else:
                contact.has_sc_sc_contact = True

            # Evidence: the closest atom pair seen from this direction, which
            # is what the .pml distance objects and the CSV distance column use.
            if contact.res1 == r1:
                if dist < contact.dist_res1_sc_to_res2:
                    contact.dist_res1_sc_to_res2 = dist
                    contact.atom_res1_sc_name = probe_names[i]
                    contact.atom_res2_target_name = target_names[j]
            else:
                if dist < contact.dist_res2_sc_to_res1:
                    contact.dist_res2_sc_to_res1 = dist
                    contact.atom_res2_sc_name = probe_names[i]
                    contact.atom_res1_target_name = target_names[j]

    for contact in contacts.values():
        pair = contact.chain_pair
        analysis.chain_pair_counts[pair] = analysis.chain_pair_counts.get(pair, 0) + 1

    for (c1, c2), count in analysis.chain_pair_counts.items():
        for a, b in ((c1, c2), (c2, c1)):
            if a not in analysis.best_partner or count > analysis.best_partner[a][1]:
                analysis.best_partner[a] = (b, count)


# --------------------------------------------------------------------------
# step 2: which chain pairs are the dimers
# --------------------------------------------------------------------------

def _classify_interfaces(analysis: '_State') -> None:
    """A chain's dimer partner is the chain it shares the most residue contacts
    with; every other pairing is an interdimer interface.

    This is the load-bearing assumption of the whole Figure 1 analysis. It is
    true for a T=3 trimer-of-dimers, where the intradimer interface is far the
    largest, and ``min_contacts_threshold`` guards the degenerate case of a
    chain with almost no contacts at all.
    """
    for chain, (partner, count) in analysis.best_partner.items():
        if count >= analysis.min_contacts_threshold:
            analysis.intra_dimer_pairs.add(tuple(sorted((chain, partner))))

    for contact in analysis.contacts.values():
        contact.interface_type = (
            'intra-dimer' if contact.chain_pair in analysis.intra_dimer_pairs
            else 'inter-dimer')


# --------------------------------------------------------------------------
# step 3: roll contacts up per residue
# --------------------------------------------------------------------------

def _summarise_residues(analysis: '_State') -> None:
    """Build the per-residue view that Figure 1 counts.

    Only a residue that contributes a side chain gets a summary: mutation acts
    on side chains, so a residue holding an interface with backbone alone is
    not a target and is deliberately absent.

    The side-chain counters below are informational and are reported as-is so
    that the published tables remain reproducible from this code. Three
    properties of them are worth stating plainly:

      1. the intradimer branch increments the ``inter_`` counters, so the
         Inter_SC_to_SC and Inter_SC_to_BB columns count side-chain contacts of
         either interface type rather than interdimer contacts alone;
      2. ``intra_sc_contacts`` is never incremented, so the Intra_SC_Contacts
         column is 0 in every output file;
      3. the two intradimer sub-branches apply different tests to the two
         residues of a contact.

    None of the three affects the classification, which reads only the partner
    sets, so the residue counts reported in the manuscript are unaffected.
    """
    residues = analysis.residues

    def summary_for(key: ResidueKey) -> '_ResidueBuilder':
        got = residues.get(key)
        if got is None:
            got = _ResidueBuilder(residue_key=key)
            residues[key] = got
        return got

    for contact in analysis.contacts.values():
        r1, r2 = contact.res1, contact.res2
        r1_probes = contact.res1_sc_found_target
        r2_probes = contact.res2_sc_found_target

        if contact.interface_type == 'intra-dimer':
            if r1_probes:
                s1 = summary_for(r1)
                s1.intra_dimer_partners.add(r2)
                if contact.has_sc_sc_contact:                 # see note (1)
                    s1.inter_sc_to_sc += 1
                elif contact.res1_sc_hit_res2_bb:
                    s1.inter_sc_to_bb += 1
            if r2_probes:
                s2 = summary_for(r2)
                s2.intra_dimer_partners.add(r1)
                if contact.res2_sc_hit_res1_bb:               # see note (3)
                    s2.inter_sc_to_bb += 1
                else:
                    s2.inter_sc_to_sc += 1
        else:
            if r1_probes:
                s1 = summary_for(r1)
                s1.inter_dimer_partners.add(r2)
                if r2_probes:
                    s1.inter_sc_to_sc += 1
                else:
                    s1.inter_sc_to_bb += 1
            if r2_probes:
                s2 = summary_for(r2)
                s2.inter_dimer_partners.add(r1)
                if r1_probes:
                    s2.inter_sc_to_sc += 1
                else:
                    s2.inter_sc_to_bb += 1

In [ ]:
%%writefile capsid/render.py
"""RENDER -- pure. Reads the InterfaceAnalysis record set and nothing else.

No file is opened here, no structure is consulted, nothing is classified. Every
view is built from the same records, and the machine-readable views (the table
row lists) are built first, with the human-readable report rendered from the
same records, so that a number on screen and a number in a file are the same
number.

Column order and value formatting in the tables are fixed by the published
data files and must not be changed.
"""

import csv
import io
import os
from typing import Dict, List, Optional, Set, Tuple

from .artifacts import Artifact, RenderedAnalysis
from .modes import HISTORICAL, Mode

from .palette import CLASS_STYLE, chain_colors
from .records import InterfaceAnalysis, ResidueSummary

CONTACT_HEADER = [
    'Interface_Type',
    'Query_Chain', 'Query_Dimer_Partner', 'Query_Resi', 'Query_Resn',
    'Query_Classification',
    'Target_Chain', 'Target_Dimer_Partner', 'Target_Resi', 'Target_Resn',
    'Target_Classification',
    'Min_Distance_A', 'Contact_Type',
]

RESIDUE_HEADER = [
    'Chain', 'Dimer_Partner', 'Resi', 'Resn', 'Classification',
    'Intra_Partner_Count', 'Inter_Partner_Count', 'Total_Partners',
    'Inter_SC_to_SC', 'Inter_SC_to_BB', 'Intra_SC_Contacts',
    'Partner_Chains',
]


# --------------------------------------------------------------------------
# machine-readable views -- built first, everything else reads these records
# --------------------------------------------------------------------------

def contact_rows(a: InterfaceAnalysis) -> List[list]:
    """One row per DIRECTION of each contact, i.e. per side chain that reached.

    A pair whose two side chains both reach the other yields two rows; a pair
    held by one side chain against a backbone yields one. This is why the row
    count exceeds the contact count.
    """
    rows = []
    ordered = sorted(a.contacts.values(),
                     key=lambda c: (c.interface_type, c.res1[0], c.res1[1]))
    for c in ordered:
        c1, i1, n1 = c.res1
        c2, i2, n2 = c.res2
        p1 = a.dimer_partner(c1) or 'None'
        p2 = a.dimer_partner(c2) or 'None'
        s1 = a.residues.get(c.res1)
        s2 = a.residues.get(c.res2)
        k1 = s1.classification if s1 else 'unknown'
        k2 = s2.classification if s2 else 'unknown'

        if c.res1_sc_found_target:
            rows.append([c.interface_type,
                         c1, p1, i1, n1, k1,
                         c2, p2, i2, n2, k2,
                         f'{c.dist_res1_sc_to_res2:.3f}', c.contact_type])
        if c.res2_sc_found_target:
            rows.append([c.interface_type,
                         c2, p2, i2, n2, k2,
                         c1, p1, i1, n1, k1,
                         f'{c.dist_res2_sc_to_res1:.3f}', c.contact_type])
    return rows


def residue_rows(a: InterfaceAnalysis) -> List[list]:
    """One row per side-chain-contributing residue, chain then number."""
    rows = []
    for s in sorted(a.residues.values(),
                    key=lambda s: (s.residue_key[0], s.residue_key[1])):
        chain, resid, resname = s.residue_key
        rows.append([
            chain, a.dimer_partner(chain) or 'None', resid, resname,
            s.classification,
            len(s.intra_dimer_partners), len(s.inter_dimer_partners),
            s.total_partners,
            s.inter_sc_to_sc, s.inter_sc_to_bb, s.intra_sc_contacts,
            ';'.join(sorted(s.partner_chains)),
        ])
    return rows


def _csv_bytes(header, rows) -> bytes:
    """Serialise to the exact byte payload that will land on disk.

    Serialisation belongs here, not in EFFECT. EFFECT's only job is to put
    finished bytes somewhere; if it were still formatting numbers or choosing
    line endings it would be making decisions, and the phase boundary would be
    a comment rather than a fact.

    csv.writer's default CRLF terminator is what the published files use.
    """
    buffer = io.StringIO(newline='')
    writer = csv.writer(buffer)
    writer.writerow(header)
    writer.writerows(rows)
    return buffer.getvalue().encode('utf-8')


def contacts_csv(a: InterfaceAnalysis) -> bytes:
    return _csv_bytes(CONTACT_HEADER, contact_rows(a))


def residues_csv(a: InterfaceAnalysis) -> bytes:
    return _csv_bytes(RESIDUE_HEADER, residue_rows(a))


# --------------------------------------------------------------------------
# human views -- same records, no second opinion
# --------------------------------------------------------------------------

def detection_report(a: InterfaceAnalysis) -> str:
    out = [f'Structure: {a.pdb_path}',
           f'Chains found: {a.chain_ids}',
           f'Cutoff: {a.cutoff} A   exclude H: {a.exclude_hydrogens}',
           '', 'Atom contacts per ordered chain pair (probe -> target):']
    for (s1, s2), n in a.atom_contact_counts.items():
        out.append(f'  {s1}.sidechain -> {s2}: {n}')
    out += ['', 'Residue contacts per chain pair:']
    for pair in sorted(a.chain_pair_counts):
        out.append(f'  {pair[0]}-{pair[1]}: {a.chain_pair_counts[pair]}')
    out.append(f'Total unique residue-residue contacts: {len(a.contacts)}')
    return '\n'.join(out)


def summary_report(a: InterfaceAnalysis) -> str:
    counts = a.class_counts()
    intra = len(a.contacts_of_type('intra-dimer'))
    inter = len(a.contacts_of_type('inter-dimer'))
    out = ['=' * 70, 'INTERFACE ANALYSIS SUMMARY', '=' * 70, '',
           f'Structure: {a.pdb_path}',
           f'Chains: {", ".join(a.chain_ids)}',
           f'Distance cutoff: {a.cutoff} A', '',
           f'Intra-dimer pairs: {sorted(a.intra_dimer_pairs)}', '',
           'Contact statistics:',
           f'  Total contacts: {len(a.contacts)}',
           f'  Intra-dimer: {intra}',
           f'  Inter-dimer: {inter}', '',
           'Residue classifications (side-chain-contributing only):',
           f"  Intra-dimer only: {counts['intra-dimer']}",
           f"  Inter-dimer only: {counts['inter-dimer']}",
           f"  Both interfaces:  {counts['both']}"]

    both = a.by_class('both')
    if both:
        out += ['', 'Residues at both interfaces:']
        for s in sorted(both, key=lambda s: (s.residue_key[0], s.residue_key[1])):
            ch, ri, rn = s.residue_key
            out.append(f'  {ch}-{rn}{ri}: {len(s.intra_dimer_partners)} intra, '
                       f'{len(s.inter_dimer_partners)} inter')
    return '\n'.join(out)


def figure1_counts(a: InterfaceAnalysis) -> Dict[str, int]:
    """The three interface-residue counts, named unambiguously.

    These are three different numbers and it is easy to quote the wrong one:
    ``interdimer_total`` counts every residue with an interdimer contact and
    therefore INCLUDES the shared residues, while ``interdimer_exclusive``
    counts only those with no intradimer contact. Returning all three under
    explicit names means a caller has to choose deliberately.
    """
    counts = a.class_counts()
    return {
        'interdimer_total': counts['inter-dimer'] + counts['both'],
        'interdimer_exclusive': counts['inter-dimer'],
        'shared_inter_and_intra': counts['both'],
    }


def mutation_report(a: InterfaceAnalysis) -> str:
    out = ['', '=' * 70, 'INTER-DIMER MUTATION ANALYSIS', '=' * 70, '',
           'Rationale:',
           '  high     - side chain reaches a partner SIDE CHAIN across dimers',
           '  moderate - side chain reaches only a partner BACKBONE',
           '  (backbone-only residues are not listed: mutation cannot affect them)']

    for potential, blurb in (('high', 'side-chain-to-side-chain contacts across dimers'),
                             ('moderate', 'side-chain contacts to partner backbones only')):
        group = a.by_potential(potential)
        if not group:
            continue
        key = ((lambda s: -s.inter_sc_to_sc) if potential == 'high'
               else (lambda s: -s.inter_sc_to_bb))
        out += ['', '-' * 70,
                f'{potential.upper()} POTENTIAL ({len(group)} residues)', blurb,
                '-' * 70,
                f"{'Residue':<12} {'SC->SC':<10} {'SC->BB':<10} Partners",
                '-' * 70]
        for s in sorted(group, key=key):
            ch, ri, rn = s.residue_key
            out.append(f'{ch}-{rn}{ri:<6} {s.inter_sc_to_sc:<10} '
                       f'{s.inter_sc_to_bb:<10} {";".join(sorted(s.partner_chains))}')
    return '\n'.join(out)


# --------------------------------------------------------------------------
# PyMOL view
# --------------------------------------------------------------------------

def _chain_selector(chain_ids: List[str]) -> str:
    """PyMOL needs 'segi' for multi-character ids and 'chain' for single."""
    return 'segi' if any(len(c) > 1 for c in chain_ids) else 'chain'


def _resid_selection(by_chain: Dict[str, Set[int]], selector: str) -> Optional[str]:
    parts = [f'({selector} {ch} and resi {"+".join(str(r) for r in sorted(rs))})'
             for ch, rs in by_chain.items() if rs]
    return ' or '.join(parts) if parts else None


def pymol_script(a: InterfaceAnalysis, output_pse: str, mode: Mode = HISTORICAL,
                 color_cycle=None, cartoon_transparency: float = 0.5) -> str:
    """Return the full .pml text. Writing it is EFFECT's job, not this one's.

    The structure loaded is always the one DECIDE analysed. There is no
    parameter to override it: a script that loaded a different structure than
    the tables describe would be a picture of one capsid captioned with the
    numbers of another, and nothing downstream would notice.

    Draws three residue classes and, for each, the exact atom pair that put the
    residue in that class -- so the picture carries the same evidence as the CSV
    rather than a redrawn approximation of it.
    """
    selector = _chain_selector(a.chain_ids)
    colors = chain_colors(a.chain_ids, color_cycle)
    load_path = a.pdb_path

    # (selection name, residues, pymol colour, legend line, section banner,
    #  optional second banner line, statistics line label)
    # The three residue classes, defined once so that the script's legend, its
    # selections, its grouping and its statistics all describe the same sets.
    groups = [
        ('inter_dimer_only', a.resids_by_class('inter-dimer'),
         CLASS_STYLE['inter-dimer']['pymol'],
         '#   Orange: Inter-dimer only residues',
         '# ===== INTER-DIMER ONLY RESIDUES (Orange) ====', None,
         'Inter-dimer only residues (orange)'),
        ('both_interfaces', a.resids_by_class('both'),
         CLASS_STYLE['both']['pymol'],
         '#   Blue (slate): Residues at both inter-dimer AND intra-dimer interfaces',
         '# ===== BOTH INTERFACE RESIDUES (Blue) ====',
         "# These residues contact both inter-dimer AND their dimer partner",
         'Both interface residues (blue)'),
        ('dimer_partner_contacts', a.dimer_partner_resids(),
         CLASS_STYLE['dimer_partner']['pymol'],
         "#   Green (splitpea): Dimer partner residues contacted by 'both' residues",
         '# ===== DIMER PARTNER RESIDUES (Bright Green) ====',
         "# These are the intra-dimer partners contacted by 'both' residues",
         'Dimer partner contacts (green)'),
    ]

    L = ['# PyMOL Inter-Dimer Interface Analysis Script',
         '# Generated by Protein Interface Analyzer',
         f'# PDB: {load_path}',
         f'# Cutoff: {a.cutoff} Angstroms',
         f'# Chain selector: {selector} (auto-detected)',
         '#', '# Color scheme:']
    for group in groups:
        L.append(group[3])
    L += ['', '# Dimer partnerships:']
    for chain in a.chain_ids:
        partner = a.dimer_partner(chain)
        if partner:
            L.append(f'#   {chain} <-> {partner} '
                     f'({a.best_partner[chain][1]} contacts)')
    L += ['',
          f'load {load_path}, structure',
          'hide everything',
          'show cartoon, structure',
          f'set cartoon_transparency, {cartoon_transparency}',
          'set stick_radius, 0.2',
          'bg_color white', '', '# Color chains']
    for chain in a.chain_ids:
        L.append(f'color {colors[chain]["pymol"]}, {selector} {chain}')
    L.append('')

    for name, by_chain, color, _legend, banner, subbanner, _stat in groups:
        L.append(banner)
        if subbanner:
            L.append(subbanner)
        selection = _resid_selection(by_chain, selector)
        if selection:
            L += [f'select {name}, {selection}',
                  f'show sticks, {name}',
                  f'color {color}, {name}', '']
        else:
            L += [f'# No {name} residues found', '']

    L += ['# ===== LABELS ====']
    if a.by_class('both'):
        L.append('label both_interfaces and name CA, "%s%s" % (resn, resi)')
    L += ['set label_size, 12', 'set label_color, white',
          'set label_outline_color, black', '', '# ===== ORGANIZE SELECTIONS ====']
    present = [g[0] for g in groups if _resid_selection(g[1], selector)]
    if present:
        L.append(f'group inter_dimer_analysis, {" ".join(present)}')
    L += ['', '# ===== STATISTICS ====']
    for _name, by_chain, _color, _legend, _b, _sb, stat in groups:
        L.append(f'# {stat}: {sum(len(r) for r in by_chain.values())}')
    L += ['', '# Finalize view', 'deselect', 'center structure',
          '', '# ===== EXACT ATOMIC DISTANCE MEASUREMENTS =====',
          ]
    # PyMOL does not accept a trailing comment on this command: it parses '#'
    # as a representation name, errors, and the labels are never hidden. The
    # historical script keeps that defect because it is what was run; the clean
    # script puts the comment on its own line so the command works.
    if mode.pml_inline_comment:
        L.append('hide labels # Hide labels initially to prevent visual clutter')
    else:
        L += ['# Hide labels initially to prevent visual clutter', 'hide labels']

    both_keys = {s.residue_key for s in a.by_class('both')}
    for c in a.contacts.values():
        c1, i1, _ = c.res1
        c2, i2, _ = c.res2
        for atom1, atom2, suffix, present_ in (
                (c.atom_res1_sc_name, c.atom_res2_target_name, 'dir1',
                 c.res1_sc_found_target),
                (c.atom_res1_target_name, c.atom_res2_sc_name, 'dir2',
                 c.res2_sc_found_target and not (
                     c.res1_sc_found_target
                     and c.atom_res1_sc_name == c.atom_res1_target_name
                     and c.atom_res2_target_name == c.atom_res2_sc_name))):
            if not present_:
                continue
            sel1 = f'{selector} {c1} and resi {i1} and name {atom1}'
            sel2 = f'{selector} {c2} and resi {i2} and name {atom2}'
            if c.interface_type == 'inter-dimer':
                dname = f'dist_inter_{c1}{i1}_{c2}{i2}_{suffix}'
                L += [f'distance {dname}, {sel1}, {sel2}',
                      f'color {CLASS_STYLE["inter-dimer"]["pymol"]}, {dname}']
            elif c.res1 in both_keys or c.res2 in both_keys:
                dname = f'dist_intra_{c1}{i1}_{c2}{i2}_{suffix}'
                L += [f'distance {dname}, {sel1}, {sel2}',
                      f'color {CLASS_STYLE["dimer_partner"]["pymol"]}, {dname}']

    L += ['', '# Group distances for easy toggling in PyMOL GUI',
          'group inter_measurements, dist_inter_*',
          'group intra_measurements, dist_intra_*', '',
          'zoom structure', 'set ray_shadows, 0', '',
          f'save {output_pse}', '# End of script']
    return '\n'.join(L)


# --------------------------------------------------------------------------
# py3Dmol view -- same three classes, same colours, same records
# --------------------------------------------------------------------------

def py3dmol_view(a: InterfaceAnalysis, pdb_text: str, pdb_sha256: str,
                 color_cycle=None, width: int = 900, height: int = 700,
                 cartoon_opacity: float = 0.6):
    """Build an interactive viewer over the structure DECIDE analysed.

    The caller supplies the coordinates because RENDER does not read files, but
    it must also supply their hash, and that hash must match the one recorded
    when the analysis was decided. Arbitrary coordinates are refused: the view
    and the tables have to describe the same structure.
    """
    # Checked before the optional import: refusing the wrong structure must not
    # depend on whether a viewer library happens to be installed.
    if pdb_sha256 != a.pdb_sha256:
        raise ValueError(
            'refusing to render a structure the analysis did not describe: '
            f'analysis was decided on sha256 {a.pdb_sha256[:16]}, '
            f'supplied coordinates are {pdb_sha256[:16]}')

    import py3Dmol

    selector = 'seg' if any(len(c) > 1 for c in a.chain_ids) else 'chain'
    colors = chain_colors(a.chain_ids, color_cycle)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_text, 'pdb')
    for chain in a.chain_ids:
        view.setStyle({selector: chain},
                      {'cartoon': {'color': colors[chain]['hex'],
                                   'opacity': cartoon_opacity}})

    for by_chain, style in ((a.resids_by_class('inter-dimer'), 'inter-dimer'),
                            (a.resids_by_class('both'), 'both'),
                            (a.dimer_partner_resids(), 'dimer_partner')):
        for chain, resids in by_chain.items():
            if resids:
                view.addStyle({selector: chain, 'resi': sorted(int(r) for r in resids)},
                              {'stick': {'color': CLASS_STYLE[style]['hex'],
                                         'radius': 0.2}})
    view.zoomTo()
    return view


# --------------------------------------------------------------------------
# the render plan -- the complete artifact set, closed before EFFECT begins
# --------------------------------------------------------------------------

def output_stem(a: InterfaceAnalysis) -> str:
    """The name every artifact for this analysis is built from.

    Naming is a RENDER decision. EFFECT is handed finished names and has
    nothing to choose.
    """
    return os.path.splitext(os.path.basename(a.pdb_path))[0] + '_interface'


def render_analysis(a: InterfaceAnalysis, include_pml: bool = True,
                    mode: Mode = HISTORICAL, color_cycle=None) -> RenderedAnalysis:
    """Turn one decision record into every view and every finished file.

    Everything the run will produce exists after this returns: the structured
    rows, the byte payloads, their names, and the human report. Nothing is left
    to be computed while files are being written.
    """
    stem = output_stem(a)
    contacts = contact_rows(a)
    residues = residue_rows(a)

    artifacts = [
        Artifact(f'{stem}_contacts.csv', 'text/csv',
                 _csv_bytes(CONTACT_HEADER, contacts)),
        Artifact(f'{stem}_residues.csv', 'text/csv',
                 _csv_bytes(RESIDUE_HEADER, residues)),
    ]
    if include_pml:
        # The PyMOL script is the one artifact whose contents depend on the
        # contract, so it is the one artifact whose name does too.
        pml_name = mode.named(stem, 'pml')
        artifacts.append(Artifact(
            pml_name, 'text/x-pymol',
            pymol_script(a, output_pse=mode.named(stem, 'pse'), mode=mode,
                         color_cycle=color_cycle).encode('utf-8')))

    report = '\n'.join([detection_report(a), summary_report(a),
                        mutation_report(a)])
    return RenderedAnalysis(
        source=a.pdb_path,
        rows={'contacts': tuple(tuple(r) for r in contacts),
              'residues': tuple(tuple(r) for r in residues)},
        artifacts=tuple(artifacts),
        report=report)

In [ ]:
%%writefile capsid/validate.py
"""VALIDATE -- between RENDER and EFFECT, and before anything is written.

The question this answers is not "are the counts the same" but "is every row
the same row". A count comparison passes a rendering that drops one row and
duplicates another, reorders rows, moves a value into the wrong column, or
changes a distance. So each check derives the ordered scientific identity three
ways -- from the decision record, from the structured rows, and by parsing the
finished bytes back -- and compares them in full.

Parsing the payload back with csv.reader rather than counting line endings also
removes the assumption that no field can contain one.
"""

import csv
import io
from typing import List, Tuple

from .artifacts import RenderedAnalysis
from .records import InterfaceAnalysis
from . import render


class ValidationError(AssertionError):
    """A rendering did not agree with the record it claims to describe."""


def _parse(payload: bytes) -> Tuple[List[str], List[List[str]]]:
    rows = list(csv.reader(io.StringIO(payload.decode('utf-8'), newline='')))
    if not rows:
        raise ValidationError('empty payload')
    return rows[0], rows[1:]


def _identities_from_records(a: InterfaceAnalysis) -> List[tuple]:
    """The ordered contact identities, derived from the record set alone.

    A separate walk from render.contact_rows: it re-derives the ordering rule
    and emits one identity per direction, without going near the renderer.
    """
    out = []
    ordered = sorted(a.contacts.values(),
                     key=lambda c: (c.interface_type, c.res1[0], c.res1[1]))
    for c in ordered:
        c1, i1, n1 = c.res1
        c2, i2, n2 = c.res2
        if c.res1_sc_found_target:
            out.append((c.interface_type, c1, str(i1), n1, c2, str(i2), n2,
                        f'{c.dist_res1_sc_to_res2:.3f}', c.contact_type))
        if c.res2_sc_found_target:
            out.append((c.interface_type, c2, str(i2), n2, c1, str(i1), n1,
                        f'{c.dist_res2_sc_to_res1:.3f}', c.contact_type))
    return out


def _identity_from_contact_row(row) -> tuple:
    r = [str(v) for v in row]
    return (r[0], r[1], r[3], r[4], r[6], r[8], r[9], r[11], r[12])


def _residue_identities_from_records(a: InterfaceAnalysis) -> List[tuple]:
    out = []
    for s in sorted(a.residues.values(),
                    key=lambda s: (s.residue_key[0], s.residue_key[1])):
        chain, resid, resname = s.residue_key
        out.append((chain, str(resid), resname, s.classification,
                    str(len(s.intra_dimer_partners)),
                    str(len(s.inter_dimer_partners))))
    return out


def _identity_from_residue_row(row) -> tuple:
    r = [str(v) for v in row]
    return (r[0], r[2], r[3], r[4], r[5], r[6])


def validate_rendering(a: InterfaceAnalysis, rendered: RenderedAnalysis,
                       expected_names=None) -> None:
    """Raise unless every view describes exactly the same analysis.

    ``expected_names`` closes the artifact set: an artifact that went missing,
    was renamed, or was added is caught here rather than discovered on disk.
    """
    if rendered.source != a.pdb_path:
        raise ValidationError(
            f'rendering describes {rendered.source!r}, record describes '
            f'{a.pdb_path!r}')

    if expected_names is not None:
        got, want = set(rendered.names), set(expected_names)
        if got != want:
            raise ValidationError(
                f'artifact set differs: missing {sorted(want - got)}, '
                f'unexpected {sorted(got - want)}')

    stem = render.output_stem(a)
    checks = (
        (f'{stem}_contacts.csv', 'contacts', render.CONTACT_HEADER,
         _identities_from_records(a), _identity_from_contact_row),
        (f'{stem}_residues.csv', 'residues', render.RESIDUE_HEADER,
         _residue_identities_from_records(a), _identity_from_residue_row),
    )

    for name, key, header, from_records, identity_of in checks:
        payload = rendered.artifact(name).payload
        parsed_header, parsed_rows = _parse(payload)
        rows = rendered.rows[key]

        if parsed_header != list(header):
            raise ValidationError(
                f'{name}: header changed\n  bytes : {parsed_header}\n'
                f'  render: {list(header)}')

        if not (len(from_records) == len(rows) == len(parsed_rows)):
            raise ValidationError(
                f'{name}: row counts disagree -- records={len(from_records)} '
                f'rendered={len(rows)} bytes={len(parsed_rows)}')

        from_rows = [identity_of(r) for r in rows]
        from_bytes = [identity_of(r) for r in parsed_rows]

        for i, (rec, row, byt) in enumerate(zip(from_records, from_rows,
                                                from_bytes), start=1):
            if not (rec == row == byt):
                raise ValidationError(
                    f'{name}: row {i} differs between derivations\n'
                    f'  records: {rec}\n  rendered: {row}\n  bytes   : {byt}')

        # full-row equality between the structured view and the bytes, so a
        # value moved into a column the identity does not cover is still caught
        for i, (row, byt) in enumerate(zip(rows, parsed_rows), start=1):
            if [str(v) for v in row] != byt:
                raise ValidationError(
                    f'{name}: row {i} full contents differ\n'
                    f'  rendered: {[str(v) for v in row]}\n  bytes   : {byt}')

In [ ]:
%%writefile capsid/effect.py
"""EFFECT -- the only phase that writes. It classifies nothing.

It iterates exactly the rows RENDER produced and puts them on disk. If EFFECT
and RENDER can ever disagree, it is because a decision leaked in here.

The invariant enforced below compares three independently derived counts rather
than calling one function twice, so that it can actually fail when two code
paths disagree.
"""

import hashlib
import os
import shutil
import tempfile
from typing import Dict, Iterable, Optional

from .artifacts import Artifact, EffectReceipt


class EffectError(RuntimeError):
    """The commit could not be performed, or did not land as planned."""


def _hashes(artifacts: Iterable[Artifact]) -> Dict[str, str]:
    return {a.relative_name: a.sha256 for a in artifacts}


def _scan(directory: str) -> Dict[str, str]:
    """Name -> sha256 for everything actually present, read back from disk."""
    found = {}
    for root, _dirs, files in os.walk(directory):
        for name in files:
            path = os.path.join(root, name)
            rel = os.path.relpath(path, directory).replace(os.sep, '/')
            with open(path, 'rb') as fh:
                found[rel] = hashlib.sha256(fh.read()).hexdigest()
    return found


def commit_artifacts(artifacts, destination: str,
                     replace_existing: bool = False) -> EffectReceipt:
    """Write a closed artifact set into ``destination``, all or nothing.

    Everything is written into a staging directory first and read back to
    confirm it landed. Only then does the destination change.
    """
    artifacts = tuple(artifacts)
    if not artifacts:
        raise EffectError('refusing to commit an empty artifact set')

    destination = os.path.abspath(destination)
    if os.path.dirname(destination) == destination:
        raise EffectError(f'refusing to write to a filesystem root: {destination}')
    if os.path.exists(destination) and not replace_existing:
        raise EffectError(
            f'{destination} already exists; pass replace_existing=True to '
            f'replace it. Refusing to mix files from two runs.')

    planned = _hashes(artifacts)
    if len(planned) != len(artifacts):
        raise EffectError('duplicate artifact names in the committed set')

    parent = os.path.dirname(destination)
    os.makedirs(parent, exist_ok=True)   # where to put it is EFFECT's business
    staging = tempfile.mkdtemp(prefix='.commit_', dir=parent)
    try:
        for artifact in artifacts:
            path = os.path.join(staging, artifact.relative_name)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            with open(path, 'wb') as fh:
                fh.write(artifact.payload)

        staged = _scan(staging)
        if staged != planned:
            raise EffectError(
                f'staged set does not match the plan: '
                f'missing {sorted(set(planned) - set(staged))}, '
                f'unexpected {sorted(set(staged) - set(planned))}')

        previous = None
        if os.path.exists(destination):
            previous = destination + '.replaced'
            shutil.rmtree(previous, ignore_errors=True)
            os.replace(destination, previous)
        try:
            os.replace(staging, destination)
        except BaseException:
            if previous is not None:            # put the old set back
                os.replace(previous, destination)
            raise
        staging = None
        if previous is not None:
            shutil.rmtree(previous, ignore_errors=True)

        committed = _scan(destination)
        if committed != planned:
            raise EffectError(
                f'committed set does not match the plan: '
                f'missing {sorted(set(planned) - set(committed))}, '
                f'unexpected {sorted(set(committed) - set(planned))}')

        return EffectReceipt(destination=destination, planned=planned,
                             staged=staged, committed=committed)
    finally:
        if staging is not None:
            shutil.rmtree(staging, ignore_errors=True)

In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")
import capsid
print("analysis package ready:", ", ".join(capsid.__all__))

## 1. DECIDE

The only step that reads the structure, and the only step that classifies anything. It returns a frozen record: nothing downstream can change what was decided here, or decide anything further.

In [ ]:
from capsid import decide

pdb = PDB_FILE or find_input("the trimer-of-dimers PDB file(s)", ["*.pdb"],
                             ["data/models", "data", "."], allow_multiple=True)
require(pdb, "the trimer-of-dimers PDB")
pdb_paths = [pdb] if isinstance(pdb, str) else list(pdb)

decisions = []
for path in pdb_paths:
    decision = decide.analyse(path, cutoff=CUTOFF,
                              exclude_hydrogens=EXCLUDE_HYDROGENS,
                              min_contacts_threshold=MIN_DIMER_CONTACTS)
    decisions.append(decision)
    print(f"{os.path.basename(path)}")
    print(f"   subunits            : {', '.join(decision.chain_ids)}")
    print(f"   dimers inferred     : "
          f"{', '.join(f'{a}-{b}' for a, b in sorted(decision.intra_dimer_pairs))}")
    print(f"   contacts            : {len(decision.contacts)}")
    print(f"   interface residues  : {len(decision.residues)}")
    print(f"   coordinates sha256  : {decision.pdb_sha256[:16]}")

## 2. RENDER

Pure. It receives the record and returns every table and every finished file. It reads nothing and writes nothing, so the numbers printed below and the bytes written later are the same numbers.

In [ ]:
from capsid import render

renderings = [render.render_analysis(d, include_pml=WRITE_PYMOL_SCRIPT)
              for d in decisions]

for decision, rendered in zip(decisions, renderings):
    counts = render.figure1_counts(decision)
    print(f"{os.path.basename(decision.pdb_path)}")
    print(f"   interdimer total     : {counts['interdimer_total']}")
    print(f"   interdimer exclusive : {counts['interdimer_exclusive']}")
    print(f"   shared with intra    : {counts['shared_inter_and_intra']}")
    print(f"   artifacts prepared   : {', '.join(rendered.names)}")
    print()

print("total     = every residue with an interdimer contact, shared ones included")
print("exclusive = residues with interdimer contacts and no intradimer contact")
print("shared    = residues contributing to both interfaces")

## 3. VALIDATE

Before anything is written, the same rows are derived three ways — from the record, from the table, and by parsing the finished bytes back — and compared row by row. The artifact set is checked against the expected names. A reordered, duplicated or edited row fails here.

In [ ]:
from capsid import validate, manifest

for decision, rendered in zip(decisions, renderings):
    stem = os.path.splitext(os.path.basename(decision.pdb_path))[0]
    validate.validate_rendering(
        decision, rendered,
        expected_names=manifest.artifact_names(stem, WRITE_PYMOL_SCRIPT))
    rows = sum(len(v) for v in rendered.rows.values())
    print(f"{stem:<26} {rows:>5} rows agree across all three derivations, "
          f"{len(rendered.names)} artifacts as expected")

print("\nIf any row differed between the record, the table and the finished")
print("bytes, this cell would have raised before anything was written.")

## 4. EFFECT

Transport only. It is handed finished bytes and cannot render, classify, or choose a filename — it does not even import the renderer. Files are staged and committed as a set, so a failed run leaves the previous output untouched.

In [ ]:
from capsid import effect

written = []
for decision, rendered in zip(decisions, renderings):
    stem = os.path.splitext(os.path.basename(decision.pdb_path))[0]
    receipt = effect.commit_artifacts(rendered.artifacts,
                                      os.path.join(OUTPUT_DIR, stem),
                                      replace_existing=True)
    print(receipt.describe())
    print()
    written += [os.path.join(receipt.destination, n)
                for n in sorted(receipt.committed)]

deliver(written)